# conv-windowing-1d — worked example 3: 1-D max-pool via NumPy sliding_window_view

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `conv-windowing-1d`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Windowing is not just for convolution — any stride-1 sliding-window reduction reuses the same `(OW, KW)` view trick. NumPy's `np.lib.stride_tricks.sliding_window_view` builds that view for you, returning shape `(..., OW, KW)` with `OW = W - KW + 1`. Reducing the trailing `KW` axis with `max` gives a stride-1 1-D max-pool.

## Worked solution

**Goal.** Compute a stride-1 1-D max-pool over a `(C, W)` array using a sliding-window view, and compare against a naive Python loop.

1. **Build the window view.** `sliding_window_view(x, window_shape=KW, axis=-1)` returns shape `(C, OW, KW)` where `OW = W - KW + 1`. It is a read-only view — no data is copied.
2. **Reduce over the window axis.** The pooled value at each output position is the maximum over its `KW`-length window, i.e. `windows.max(axis=-1)`, giving `(C, OW)`.
3. **Why `axis=-1` for the view but `axis=-1` for the max differ in meaning.** The view appends a *new* trailing `KW` axis; the original `W` axis becomes `OW`. So after viewing, axis `-1` is the kernel axis we reduce.
4. **Verification.** A brute-force loop computes `max(x[:, i:i+KW])` for each `i`; the vectorized result must match it exactly (same dtype, no floating drift since it's just selection).

In [ ]:
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view

def maxpool1d_window(x: np.ndarray, KW: int) -> np.ndarray:
    windows = sliding_window_view(x, window_shape=KW, axis=-1)
    return windows.max(axis=-1)

np.random.seed(0)
x = np.random.randn(3, 10)
KW = 3
pooled = maxpool1d_window(x, KW)
OW = x.shape[-1] - KW + 1
brute = np.stack([x[:, i:i + KW].max(axis=-1) for i in range(OW)], axis=-1)
print('pooled shape:', pooled.shape)
print('matches brute force:', np.allclose(pooled, brute))